<div style="width: 100%; clear: both;">
    <div style="float: left; width: 50%;">
       <img src="https://www.uoc.edu/portal/system/modules/edu.uoc.presentations/resources/img/branding/logo-uoc-default.png", align="left" width="125px">
    </div>
</div>

<div style="float: right; width: 50%;">
    <p style="margin: 0; padding-top: 22px; text-align:right;">M2.851 - Tipología y ciclo de vida de los datos</p>
    <p style="margin: 0; text-align:right;">Grau en Ciència de Dades Aplicada</p>
    <p style="margin: 0; text-align:right; padding-button: 100px;">Estudis d'Informàtica, Multimèdia i Telecomunicació</p>
</div>
</div>

# M2.851 - Tipología y ciclo de vida de los datos
## Práctica 1 - Dataset de tecnologías utilizadas en proyectos de hardware abierto certificado
============================

Para hacer esta practica se utilizará la web de [OSHWA](https://certification.oshwa.org/list.html) (Open Source Hardware Association) como fuente principal de datos. OSHWA ofrece un programa de certificación enfocada a hardware abierto. La certificacion se obtiene si el proyecto cumple con los diferentes criterios y además otras personas verifican que el proyecto aparece en proyectos reales de software y hardware abierto. Esta web prepara los programas base sobre los cuales se puede usar como base para futuros trabajos.

La idea es ir un poco más allá de obtener simplemente el listado de proyectos que hay publicados sino construir un dataset que para poder conocesr que tecnologías aparecen en proyectore reales de hardware abierto. Un posible uso de este dataset seria utilizarlo como base para proyectos posteriores de recomentacion o seleccion de tecnologías y componentes. 

La idea general es la sigiente: Entramos en el directorio de OSHWA y obtenemos la lista de proyectos certificados. Entramos en cada ficha y extraemos los metadatos necesarios para detectar la tecnologia que nos interesa y luego construir el dataset.

Posibles usos que se les quiere dar a este dataset:
- saber que microcontrolador aparece con más frecuencia en proyectos de hardware abierto
- saber que tecnología de comunicación suelen utilizarse juntas
- que tecnologías se usan en proyectos de electrónica, robótica, domótica,IoT, etc.
- descubrir cómo influye el tiempo en la aparición de nuevas tecnologías y la desaparición de otras
- si tenemos unos requisitos, encontrar esos proyectos que se pueden utilizar para hacer recomendaciones de tecnologías y componentes.

En esta práctica nos cenraremos en la captura de la informacion haciendo webscraping y no entrenaremos ningún modelo, solo prepararemos un dataset. Tal como nos pide la practica, no utilizaremos la API oficial de OSHWA.

Principalmente se utilizaran las siguientes fuentes:

- OSHWA - About:  https://oshwa.org/about/
- Open Source Hardware Basics: https://certification.oshwa.org/basics.html
- OSHWA Certification - Documentation: https://certification.oshwa.org/process/documentation.html
- OSHWA Certification FAQ:https://certification.oshwa.org/basics/faq.html
- Geeksforgeeks (ejemplos de Web Scraping): https://www.geeksforgeeks.org/
- Stackoverflow (ejemplos de Web Scraping): https://stackoverflow.com/
- documentación oficial de las diferentes librerias que se utilizarán

## Carga de librerías

In [1]:
import re # Regular expressions
import time 
import random 
import requests # peticiones HTTP
import pandas as pd
import bs4 # BeautifulSoup para analizar el contenido HTML

from bs4 import BeautifulSoup
from urllib.parse import urljoin, unquote
from urllib.robotparser import RobotFileParser

print("Librerías importadas")
print("requests:", requests.__version__)
print("BeautifulSoup:", bs4.__version__)
print("pandas:", pd.__version__)

Librerías importadas
requests: 2.34.2
BeautifulSoup: 4.15.0
pandas: 2.3.3


## Configuración inicial de la práctica

La práctica principalmente nos pide crear un script propio de webscraping que pueda extraer informacion de una web real y generar un dataset útil para un posible análisis posterior. El mismo código debe poder descubrir enalces y contar con capacidad de navegación autónoma ademas de realizar un uso responsable del sitio web.

Para hacer la práctica se decidió utilizar pocos proyectos (al menos al principio) de los que hay en la web ya que es más fácil controlar y depurar el código. También se decidió enfocarnos únicamente en proyectos de `electrónica` para acotar el alcance de la práctica. Se dejará preparado el scraper para que pueda ser utilizado en el futuro para otros tipos de proyectos.

Entonces, nuestro codigo sera capaz de:
- aumentar el número de proyectos a analizar (con MAX_PROYECTOS)
- poder procesar todos los proyectos seleccionados (con MAX_PROYECTOS = None)
- poder extraer otros tipos de proyectos diferentes de `electrónica` (con SOLO_ELECTRONICA = False)
- poder comprobar que User-Agent está utilizando el scraper y que no es bloqueado por la web (con USER_AGENT)


In [2]:
# Bibliografía:
#              - https://requests.readthedocs.io/en/latest/user/advanced/
#              - https://www.geeksforgeeks.org/python/how-to-scrape-multiple-pages-of-a-website-using-python/
#              - https://stackoverflow.com/questions/33135929/accessing-tabular-data-via-hyperlinks-with-beautifulsoup

BASE_URL = "https://certification.oshwa.org"
URL_DIRECTORIO = BASE_URL + "/list.html"
ROBOTS_URL = BASE_URL + "/robots.txt"

# User-Agent identificable. No intentamos hacernos pasar por un navegador.
USER_AGENT = "M2.851-Scraper/0.1"

TIMEOUT = 30
ESPERA_MIN = 1.5 # Espera entre peticiones (tiempo en segundos)
ESPERA_MAX = 3.0 # Espera entre peticiones (tiempo en segundos)
SOLO_ELECTRONICA = True
# MAX_PROYECTOS = 20 solo para pruebas
MAX_PROYECTOS = 1000 # Número máximo de proyectos a descargar. None = todos los proyectos

print("Directorio:", URL_DIRECTORIO)
print("User-Agent:", USER_AGENT)
print("Máximo de proyectos en esta prueba:", MAX_PROYECTOS)


Directorio: https://certification.oshwa.org/list.html
User-Agent: M2.851-Scraper/0.1
Máximo de proyectos en esta prueba: 1000


## 1. Comprobación de robots.txt

Antes de hacer el scraping comprobamos robots.txt. No utilizaremos `RobotFileParser()` sino que descargamos el archivo con `requests` y si existe se lo pasamos al método `parse()`. Con esto podemos utilizar el mismo `UserAgent` que hemos definido.

In [3]:
# Bibliografía:
#              - https://stackoverflow.com/questions/37934316/change-user-agent-used-with-robotparser-in-python
#              - https://docs.python.org/3/library/urllib.robotparser.html

robot_parser = None
robots_estado = None

try:
    respuesta_robots = requests.get(ROBOTS_URL, headers={"User-Agent": USER_AGENT}, timeout=TIMEOUT)
    robots_estado = respuesta_robots.status_code
    print("Código robots.txt:", robots_estado)

    if robots_estado == 200:
        robot_parser = RobotFileParser()
        robot_parser.set_url(ROBOTS_URL)
        robot_parser.parse(respuesta_robots.text.splitlines())
        print("robots.txt encontrado")
        print("Disponible:",robot_parser.can_fetch(USER_AGENT, URL_DIRECTORIO))

    elif robots_estado == 404:
        print("robots.txt no encontrado (error 404).")
    else:
        print("No se puede comprobar robots.txt, hay que revisar el resultado.")

except requests.RequestException as error:
    raise RuntimeError("No se ha podido consultar robots.txt; extracción detenida") from error

Código robots.txt: 200
robots.txt encontrado
Disponible: True


## 2. Crear una sesión de requests y descarga

La idea es simple: creamos una sesión con `requests.Session()` para reutilizar las cabeceras y configuraciones en las siguientes peticiones. Comprobamos robots.txt, añadimos una espera areatoria y realizamos la peticion con `timeout` para evitar que el scraper se quede colgado. Si la respuesta es correcta, devolvemos el contenido de la página. Si la respuesta es 403 (Forbidden) o 429 (Too Many Requests) paramos el scraper. 

In [4]:
# Bibliografía:
#              - https://stackoverflow.com/questions/33135929/accessing-tabular-data-via-hyperlinks-with-beautifulsoup
#              - https://www.geeksforgeeks.org/python/how-to-scrape-multiple-pages-of-a-website-using-python/
#              - https://www.geeksforgeeks.org/python/web-scraping-for-stock-prices-in-python/
#              - https://requests.readthedocs.io/en/latest/user/advanced/

sesion = requests.Session()
sesion.headers.update({"User-Agent": USER_AGENT})

def descargar_pagina(url, esperar=True):
    """Descarga una página respetando robots.txt cuando esté disponible."""

    # Si robots.txt existe y prohíbe la URL, no hacemos la petición.
    if robot_parser is not None:
        if not robot_parser.can_fetch(USER_AGENT, url):
            raise PermissionError("robots.txt no permite acceder a: " + url)

    # Si robots.txt ha devuelto un error distinto de 200/404 mejor paramos.
    if robots_estado not in (None, 200, 404):
        raise RuntimeError("No se ha podido validar robots.txt. Estado: " + str(robots_estado))

    if esperar:
        tiempo_espera = random.uniform(ESPERA_MIN, ESPERA_MAX)
        time.sleep(tiempo_espera)

    respuesta = sesion.get(url, timeout=TIMEOUT)

    print("HTTP", respuesta.status_code, "-", url)

    if respuesta.status_code == 403:
        raise RuntimeError("Error 403 (Forbidden). Scraping detenido")

    if respuesta.status_code == 429:
        raise RuntimeError("Error 429 (Too Many Requests). Scraping detenido")

    respuesta.raise_for_status()
    return respuesta

## 3. Descargamos el directorio + inspección general

El directorio de OSHWA es una tabla con los proyectos certificados que descargaremos y procesaremos comprobando que la pagina se descarga, que encontramos la tabla y que la tabla contiene varias filas. Con esto nos aseguramos que el scraper funciona correctamente y controlamos si la web ha cambiado su estructura para saber si el scraper necesita modificaciones.

In [5]:
# Descargamos el directorio
# Bibliografía:
#              - https://www.geeksforgeeks.org/python/parsing-tables-and-xml-with-beautifulsoup/
#              - https://www.geeksforgeeks.org/python/implementing-web-scraping-python-beautiful-soup/

respuesta_directorio = descargar_pagina(URL_DIRECTORIO, esperar=False)

soup_directorio = BeautifulSoup(respuesta_directorio.content, "html.parser")

print("Título HTML:")
print(soup_directorio.title.get_text(" ", strip=True) if soup_directorio.title else "No encontrado")

tabla = soup_directorio.find("table")
if tabla is None:
    print("No se ha encontrado ninguna tabla. Hay que revisar la estructura HTML.")
else:
    filas = tabla.find_all("tr")
    print("Tabla encontrada")
    print("Número de filas encontradas:", len(filas))

HTTP 200 - https://certification.oshwa.org/list.html
Título HTML:
OSHWA Certified Projects List
Tabla encontrada
Número de filas encontradas: 3463


## 4. Extraemos los proyectos de la tabla

Despues de la inspección general sabemos que cada fila tiene la información básica del proyecto: nombre, UID, tipo principal, fecha certificación y el enlace a la ficha individual. 

Podemos recorrer las filas con `find_all('tr')` y extraer la información de cada celda con `find_all('td')`. Después podemos buscar el primer enlace `<a href ="/ficha">` que nos llevará a la ficha individual del proyecto.

In [6]:
# Bibliografía:
#              - https://www.geeksforgeeks.org/python/parsing-tables-and-xml-with-beautifulsoup/
#              - https://stackoverflow.com/questions/57066394/trying-to-grab-just-the-first-href-in-each-table-row
#              - https://stackoverflow.com/questions/75904265/python-web-scraper-not-populating-txt-file-with-any-scraped-data

def extraer_proyectos_directorio(soup):
    """
    Recorre la tabla principal de OSHWA y devuelve una lista de diccionarios.
    """
    # Lo hacemos con el patrón tabla -> filas -> celdas -> enlace tal como está en OSHWA.

    tabla = soup.find("table")
    if tabla is None: return []

    proyectos = []

    for fila in tabla.find_all("tr"):
        celdas = fila.find_all("td")

        # La cabecera no tiene td y algunas filas no válidas podrían no tener las 4 celdas esperadas.
        if len(celdas) < 4: continue
        enlace = celdas[0].find("a", href=True)
        if enlace is None: continue
        nombre = enlace.get_text(" ", strip=True)
        uid = celdas[1].get_text(" ", strip=True)
        tipo_principal = celdas[2].get_text(" ", strip=True)
        fecha_certificacion = celdas[3].get_text(" ", strip=True)

        # Los UID de OSHWA siguen un formato similar a US000001, ES000010, etc.
        # Lo utilizo como comprobación para evitar capturar enlaces de navegación.
        if not re.fullmatch(r"[A-Z]{2}\d{6}", uid): continue
        url_ficha = urljoin(URL_DIRECTORIO, enlace["href"])
        proyectos.append({"project_name_list": nombre, "uid": uid, "primary_project_type": tipo_principal, "certification_date_list": fecha_certificacion, "detail_url": url_ficha})
    return proyectos

In [7]:
# Y ahora comprobamos la función con el directorio descargado
proyectos_directorio = extraer_proyectos_directorio(soup_directorio)
print("Número de proyectos encontrados:", len(proyectos_directorio))

df_directorio = pd.DataFrame(proyectos_directorio)
display(df_directorio.head(10))

Número de proyectos encontrados: 3462


,project_name_list,uid,primary_project_type,certification_date_list,detail_url
0,OpenLog,US000001,Electronics,"October 01, 2016",https://certification.oshwa.org/us000001.html
1,blink(1) USB notification light,US000051,Electronics,"October 04, 2016",https://certification.oshwa.org/us000051.html
2,BeagleBone Black Wireless,US000011,Electronics,"October 06, 2016",https://certification.oshwa.org/us000011.html
3,ColorHug,UK000001,Electronics,"August 20, 2018",https://certification.oshwa.org/uk000001.html
4,HackEEG,US000027,Electronics,"October 08, 2016",https://certification.oshwa.org/us000027.html
5,Smoothieware,FR000001,Electronics,"October 08, 2016",https://certification.oshwa.org/fr000001.html
6,SAV MAKER I,ES000002,Electronics,"October 08, 2016",https://certification.oshwa.org/es000002.html
7,OpenEVSE,US000028,Electronics,"October 08, 2016",https://certification.oshwa.org/us000028.html
8,Open.Theremin,CH000001,Sound,"October 08, 2016",https://certification.oshwa.org/ch000001.html
9,OneRNG,NZ000001,Electronics,"October 08, 2016",https://certification.oshwa.org/nz000001.html


## 4. Filtrar proyectos de electrónica

Como ya se introdujo al principio, queremos simplificar el scraper y consideramos que únicamente nos interesan los proyectos de `electrónica`. Esto es bastante directo porque el mismo directorio tiene el campo de `Project Type` que nos permite filtrar los proyectos de `Electronics`. El código de extracción del directorio extrae todos los proyectos y aqui solamente filtramos.

In [8]:
if SOLO_ELECTRONICA:
    proyectos_seleccionados = []

    for proyecto in proyectos_directorio:
        if proyecto["primary_project_type"] == "Electronics":
            proyectos_seleccionados.append(proyecto)
else:
    proyectos_seleccionados = proyectos_directorio.copy()

print("Proyectos disponibles después del filtro:", len(proyectos_seleccionados))

# Para las pruebas trabajamos con la lista reducida.
if MAX_PROYECTOS is not None:
    proyectos_seleccionados = proyectos_seleccionados[:MAX_PROYECTOS]
print("Proyectos procesados:", len(proyectos_seleccionados))
pd.DataFrame(proyectos_seleccionados).head()

Proyectos disponibles después del filtro: 2467
Proyectos procesados: 1000


,project_name_list,uid,primary_project_type,certification_date_list,detail_url
0,OpenLog,US000001,Electronics,"October 01, 2016",https://certification.oshwa.org/us000001.html
1,blink(1) USB notification light,US000051,Electronics,"October 04, 2016",https://certification.oshwa.org/us000051.html
2,BeagleBone Black Wireless,US000011,Electronics,"October 06, 2016",https://certification.oshwa.org/us000011.html
3,ColorHug,UK000001,Electronics,"August 20, 2018",https://certification.oshwa.org/uk000001.html
4,HackEEG,US000027,Electronics,"October 08, 2016",https://certification.oshwa.org/us000027.html


## 5. Consultar la ficha individual de un proyecto

Hasta aquí hemos trabajado únicamente con la tabla principal pero uno de los objetivos de la práctica es la navegación autónoma por los enlaces de la web. Así podremos utilizar los enlaces del directorio para entrar automáticamente en la página individual de cada proyecto.

Primero hacemos la prueba con una única ficha, por ejemplo del primer proyecto de la lista.

In [9]:
# Bibliografía:
#              - https://stackoverflow.com/questions/70795634/how-can-beautifulsoup-scrape-the-pages-inside-this-list-of-hyperlinks
#              - https://stackoverflow.com/questions/33135929/accessing-tabular-data-via-hyperlinks-with-beautifulsoup
#              - https://stackoverflow.com/questions/29421840/how-to-scrape-a-page-with-beautifulsoup-and-python

if proyectos_seleccionados:
    proyecto_prueba = proyectos_seleccionados[0]

    print("Proyecto:", proyecto_prueba["project_name_list"])
    print("Ficha:", proyecto_prueba["detail_url"])

    respuesta_ficha = descargar_pagina(proyecto_prueba["detail_url"])
    soup_ficha = BeautifulSoup(respuesta_ficha.content, "html.parser")

    print("\nTítulo de la ficha:")
    print(soup_ficha.find("h1").get_text(" ", strip=True) if soup_ficha.find("h1") else "No encontrado")
    print("\nEncabezados encontrados:")

    for encabezado in soup_ficha.find_all(["h2", "h3", "h4", "h5", "h6"]):
        print(encabezado.name, "-", encabezado.get_text(" ", strip=True))
else:
    print("No hay proyectos seleccionados.")

Proyecto: OpenLog
Ficha: https://certification.oshwa.org/us000001.html
HTTP 200 - https://certification.oshwa.org/us000001.html

Título de la ficha:
OpenLog

Encabezados encontrados:
h2 - Nathan Seidle mail
h3 - OSHWA UID
h3 - Certification Date
h3 - Country
h3 - Version
h5 - Licenses
h6 - Hardware
h6 - Software
h6 - Documentation
h5 - Certification mark
h5 - Project Type
h5 - Keywords


## 7. Extracción de información de la ficha individual

Si inspecionamos una ficha individual podemos ver que tienen la siguiente estrustura: 
- OSHWA UID
- Certification Date
- Country
- Version
- Previous Version
- License
- Project Type
- Keywords

Con BeautifulSoup podemos localizar un encabezado y recorrer luego los diferentes elementos hasta llegar al siguiente encabezado. Definimos las funciones `normalizar_texto()`, `buscar_titulo()`, `textos_hasta_siguiente_titulo()` y `primer_valor_despues_de_titulo()` para poder extraer la información de cada ficha individual.

In [10]:
# Bibliografía:
#              - https://stackoverflow.com/questions/27193884/how-to-grab-alternating-child-tags-in-python-beautifulsoup
#              - https://www.geeksforgeeks.org/python/beautifulsoup-nextsibling/
#              - https://stackoverflow.com/questions/62104956/how-can-i-extract-text-and-lists-with-beautifulsoup

TAGS_TITULO = {"h1", "h2", "h3", "h4", "h5", "h6"}

def normalizar_texto(texto):
    """Elimina espacios repetidos y pasa el texto a minúsculas para comparar."""
    if texto is None: return ""
    return re.sub(r"\s+", " ", texto).strip().casefold()

def buscar_titulo(soup, texto):
    """Busca un encabezado h1-h6 cuyo texto coincida con el indicado."""
    objetivo = normalizar_texto(texto)

    for tag in soup.find_all(TAGS_TITULO):
        texto_tag = normalizar_texto(tag.get_text(" ", strip=True))
        if texto_tag == objetivo:
            return tag
    return None

def textos_hasta_siguiente_titulo(titulo):
    """
    Obtiene textos situados después de un encabezado hasta encontrar otro h1-h6.
    Se utilizan únicamente elementos sin hijos HTML para reducir textos duplicados.
    """
    if titulo is None: return []
    textos = []
    for elemento in titulo.find_all_next():
        if elemento is titulo: continue
        if elemento.name in TAGS_TITULO: break
        if elemento.name in {"script", "style"}: continue

        # Si el elemento contiene otras etiquetas, normalmente es un contenedor y su texto aparecerá de nuevo en los elementos hijos.
        if elemento.find(True) is not None: continue
        texto = elemento.get_text(" ", strip=True)
        if texto and texto not in textos:
            textos.append(texto)
    return textos

def primer_valor_despues_de_titulo(soup, texto_titulo):
    """Devuelve el primer valor textual encontrado después de un encabezado."""
    titulo = buscar_titulo(soup, texto_titulo)
    valores = textos_hasta_siguiente_titulo(titulo)
    if valores: return valores[0]
    return None

## 8. Extracción de los campos de una ficha individual

Llegados aqui podemos agrupar toda la información de la ficha en un diccionario con campos de diferentes tipos:
- texto: nombre, autor, descripción
- categoricos: pais y licencia
- fecha: fecha de certificación
- listas: tipos de proyecto y palabras clave y versiones anteriores
- URLs: web del proyecto, dcumentación, archivos gráficos del certificado, etc.

Con esto obtenemos un dataset bastante heterogéneo sin limpiar nada todavia. También se seguiran las recomendaciones de los ejemplos encontrados y utilizaremos `urljoin()` en vez de `href`ya que estos pueden ser absolutos o relativos.

A continuación, se describe cómo se realiza el proceso de extracción de datos para cada campo.

In [11]:
# Bibliografía:
#              - https://stackoverflow.com/questions/69960877/python-beautifulsoup-how-can-i-get-full-link-from-href-attribute
#              - https://www.geeksforgeeks.org/python/beautifulsoup-scraping-link-from-html/

TIPOS_PROYECTO_OSHWA = {"3d printing", "agriculture", "arts", "education", "electronics", "enclosure", "environmental", "home connection", "iot", "manufacturing", "other", "robotics", "science", "sound", "space", "tool", "wearables"}

def buscar_enlace_por_texto(soup, texto):
    """Busca un enlace cuyo texto contenga la cadena indicada."""
    texto_buscado = texto.casefold()
    for enlace in soup.find_all("a", href=True):
        texto_enlace = enlace.get_text(" ", strip=True).casefold()
        if texto_buscado in texto_enlace:
            return urljoin(BASE_URL, enlace["href"])
    return None


def extraer_descripcion(soup):
    """
    En las fichas de OSHWA la descripción aparece después de Country y antes de Version.
    Seleccionamos el texto más largo de ese bloque descartando el primer elemento, que normalmente es el nombre del país.
    """
    titulo_country = buscar_titulo(soup, "Country")
    valores = textos_hasta_siguiente_titulo(titulo_country)

    if len(valores) <= 1: return None
    candidatos = valores[1:]
    return max(candidatos, key=len) if candidatos else None


def extraer_tipos_proyecto(soup):
    """Recoge los tipos situados entre Project Type y Keywords."""
    titulo = buscar_titulo(soup, "Project Type")

    if titulo is None: return []

    tipos = []
    for elemento in titulo.find_all_next():
        if elemento is titulo: continue
        if elemento.name in TAGS_TITULO:
            texto_encabezado = normalizar_texto(elemento.get_text(" ", strip=True))
            if texto_encabezado == "keywords": break
            continue
        if elemento.find(True) is not None: continue

        texto = elemento.get_text(" ", strip=True)
        texto_normalizado = normalizar_texto(texto)
        if texto_normalizado in TIPOS_PROYECTO_OSHWA and texto not in tipos:
            tipos.append(texto)
    return tipos


def extraer_keywords(soup):
    """Recoge el texto situado después de Keywords y lo separa por comas."""
    titulo = buscar_titulo(soup, "Keywords")
    valores = textos_hasta_siguiente_titulo(titulo)
    if not valores: return []

    # Normalmente OSHWA muestra las keywords como una cadena separada por comas.
    texto_keywords = valores[0]
    return [keyword.strip() for keyword in texto_keywords.split(",") if keyword.strip()]


def extraer_marcas_certificacion(soup, url_actual):
    """Busca enlaces SVG y PNG dentro del bloque Certification mark."""
    titulo = buscar_titulo(soup, "Certification mark")

    if titulo is None: return None, None
    svg_url = None
    png_url = None

    for elemento in titulo.find_all_next():
        if elemento is titulo: continue
        if elemento.name in TAGS_TITULO:
            if normalizar_texto(elemento.get_text(" ", strip=True)) == "project type": break
        if elemento.name == "a" and elemento.get("href"):
            enlace = urljoin(url_actual, elemento["href"])
            if re.search(r"\.svg(?:\?|$)", enlace, flags=re.I):
                svg_url = svg_url or enlace
            if re.search(r"\.png(?:\?|$)", enlace, flags=re.I):
                png_url = png_url or enlace
    return svg_url, png_url

In [12]:
def extraer_detalle_oshwa(soup, url_actual):
    """Extrae la información de una ficha individual de certificación OSHWA."""
    # Nombre del proyecto
    h1 = soup.find("h1")
    project_name = h1.get_text(" ", strip=True) if h1 else None

    # Autor / organización
    h2 = soup.find("h2")
    author = h2.get_text(" ", strip=True) if h2 else None

    # Excluimos enlaces mailto antes de obtener el nombre del autor.
    if h2:
        for mail in h2.find_all("a", href=re.compile(r"^mailto:", re.I)):
            mail.decompose()
        author = h2.get_text(" ", strip=True).strip(" ()-")

    # Valores organizados por encabezados
    certification_date = primer_valor_despues_de_titulo(soup, "Certification Date")
    country = primer_valor_despues_de_titulo(soup, "Country")
    version = primer_valor_despues_de_titulo(soup, "Version")

    # Versiones anteriores no aparecen en todos los proyectos.
    previous_text = primer_valor_despues_de_titulo(soup, "Previous Versions")
    previous_versions = []

    if previous_text:
        previous_versions = re.findall(r"[A-Z]{2}\d{6}", previous_text)

    # Licencias
    hardware_license = primer_valor_despues_de_titulo(soup, "Hardware")
    software_license = primer_valor_despues_de_titulo(soup, "Software")
    documentation_license = primer_valor_despues_de_titulo(soup, "Documentation")

    # Enlaces
    project_website = buscar_enlace_por_texto(soup, "Project Website")
    documentation_url = buscar_enlace_por_texto(soup, "View Documentation")

    # Descripción, tipos y keywords
    description = extraer_descripcion(soup)
    project_types = extraer_tipos_proyecto(soup)
    keywords = extraer_keywords(soup)

    # Recursos gráficos del certificado
    certification_svg_url, certification_png_url = extraer_marcas_certificacion(soup, url_actual)

    return {
        "project_name": project_name,
        "author": author,
        "certification_date": certification_date,
        "country": country,
        "description": description,
        "version": version,
        "previous_versions": previous_versions,
        "hardware_license": hardware_license,
        "software_license": software_license,
        "documentation_license": documentation_license,
        "project_types": project_types,
        "keywords": keywords,
        "project_website": project_website,
        "documentation_url": documentation_url,
        "certification_svg_url": certification_svg_url,
        "certification_png_url": certification_png_url
    }

# 9. Probamos la extracción de una única ficha individual

Se trabaja con una única ficha ya que es más fácil hacer pruebas y depurar el código. No se elimina en el documento final ya que esta sección nos puede servir para comprobar si alguno de los encabezados o sectores ha cambiado. 

In [13]:
if proyectos_seleccionados:
    proyecto_prueba = proyectos_seleccionados[0]
    respuesta_ficha = descargar_pagina(proyecto_prueba["detail_url"])
    soup_ficha = BeautifulSoup(respuesta_ficha.content, "html.parser")
    detalle_prueba = extraer_detalle_oshwa(soup_ficha, proyecto_prueba["detail_url"])

    print("Datos obtenidos de la ficha:\n")
    for campo, valor in detalle_prueba.items():
        print(campo, ":", valor)
else:
    print("No hay proyectos disponibles para realizar la prueba.")

HTTP 200 - https://certification.oshwa.org/us000001.html
Datos obtenidos de la ficha:

project_name : OpenLog
author : Nathan Seidle
certification_date : October 01, 2016
country : United States of America
description : The SparkFun OpenLog is an open source data logger that works over a simple serial connection and supports microSD cards up to 32GB.
version : 15
previous_versions : []
hardware_license : Other
software_license : Other
documentation_license : CC BY-SA
project_types : ['Electronics']
keywords : ['Arduino']
project_website : https://www.sparkfun.com/products/13712
documentation_url : https://learn.sparkfun.com/tutorials/openlog-hookup-guide
certification_svg_url : None
certification_png_url : None


## 10. Extracción de todos los proyectos de electrónica seleccionados

Hemos comprobado que el scraper funciona con un única ficha asi que podemos ir con las demás fichas de los proyectos selecionados. Tambien aqui se comprueba la navegación autónoma ya que tiene que recorrer la lista de enlace sin que nosotros tengamos que intervenir. El programa lee los enlaces desde el directorio, entra en cada uno, encuentra las fichas y extrae la informacion y luego combina los capos del listado con los de la ficha individula. 

In [ ]:
# Bibliografía:
#              - https://stackoverflow.com/questions/70795634/how-can-beautifulsoup-scrape-the-pages-inside-this-list-of-hyperlinks
#              - https://stackoverflow.com/questions/29421840/how-to-scrape-a-page-with-beautifulsoup-and-python
#              - https://www.geeksforgeeks.org/python/web-scraping-for-stock-prices-in-python/

def ejecutar_scraping(proyectos):
    """
    Recorre las fichas descubiertas automáticamente en el directorio
    y devuelve una lista de registros.
    """
    resultados = []
    total = len(proyectos)
    for numero, proyecto in enumerate(proyectos, start=1):
        print("\n" + "=" * 80)
        print(f"Proyecto {numero}/{total}")
        print(proyecto["uid"], "-", proyecto["project_name_list"])

        try:
            respuesta = descargar_pagina(proyecto["detail_url"])
            soup = BeautifulSoup(respuesta.content, "html.parser")
            detalle = extraer_detalle_oshwa(soup, proyecto["detail_url"])
            # Unimos los campos obtenidos del listado con los obtenidos en la ficha.
            registro = {**proyecto, **detalle}
            resultados.append(registro)
            
        except (PermissionError, RuntimeError):
            # Un bloqueo o una prohibición detiene toda la extracción.
            raise
        except requests.RequestException as error:
            print("No se ha podido procesar el proyecto:", error)
    return resultados

In [15]:
# Ejecutamos el scraping
datos_oshwa = ejecutar_scraping(proyectos_seleccionados)
print("\nNúmero de fichas procesadas correctamente:", len(datos_oshwa))


Proyecto 1/1000
US000001 - OpenLog
HTTP 200 - https://certification.oshwa.org/us000001.html

Proyecto 2/1000
US000051 - blink(1) USB notification light
HTTP 200 - https://certification.oshwa.org/us000051.html

Proyecto 3/1000
US000011 - BeagleBone Black Wireless
HTTP 200 - https://certification.oshwa.org/us000011.html

Proyecto 4/1000
UK000001 - ColorHug
HTTP 200 - https://certification.oshwa.org/uk000001.html

Proyecto 5/1000
US000027 - HackEEG
HTTP 200 - https://certification.oshwa.org/us000027.html

Proyecto 6/1000
FR000001 - Smoothieware
HTTP 200 - https://certification.oshwa.org/fr000001.html

Proyecto 7/1000
ES000002 - SAV MAKER I
HTTP 200 - https://certification.oshwa.org/es000002.html

Proyecto 8/1000
US000028 - OpenEVSE
HTTP 200 - https://certification.oshwa.org/us000028.html

Proyecto 9/1000
NZ000001 - OneRNG
HTTP 200 - https://certification.oshwa.org/nz000001.html

Proyecto 10/1000
MX000001 - Byteman
HTTP 200 - https://certification.oshwa.org/mx000001.html

Proyecto 11/1000

## 11. Creamos el DataFrame

Hasa ahora hemos ido guardando cada proyecto en un diccionario y todos los diccionarios en una lista. Ahora utilziamos directamente esa lista de diccionarios para crear un DataFrame de Pandas.

In [16]:
# Bibliografía:
#              - https://www.geeksforgeeks.org/python/create-a-pandas-dataframe-from-list-of-dicts/
#              - https://www.geeksforgeeks.org/python/implementing-web-scraping-python-beautiful-soup/

df_oshwa = pd.DataFrame(datos_oshwa)
print("Número de registros:", len(df_oshwa))
print("Número de columnas:", len(df_oshwa.columns))
display(df_oshwa.head())

Número de registros: 1000
Número de columnas: 21


,project_name_list,uid,primary_project_type,certification_date_list,detail_url,project_name,author,certification_date,country,description,...,previous_versions,hardware_license,software_license,documentation_license,project_types,keywords,project_website,documentation_url,certification_svg_url,certification_png_url
0,OpenLog,US000001,Electronics,"October 01, 2016",https://certification.oshwa.org/us000001.html,OpenLog,Nathan Seidle,"October 01, 2016",United States of America,The SparkFun OpenLog is an open source data lo...,...,[],Other,Other,CC BY-SA,[Electronics],[Arduino],https://www.sparkfun.com/products/13712,https://learn.sparkfun.com/tutorials/openlog-h...,None,None
1,blink(1) USB notification light,US000051,Electronics,"October 04, 2016",https://certification.oshwa.org/us000051.html,blink(1) USB notification light,ThingM,"October 04, 2016",United States of America,blink(1) is a programmable USB notification li...,...,[],Other,Other,CC BY-SA,[Electronics],"[USB, light]",https://blink1.thingm.com/,https://github.com/todbot/blink1/blob/master/L...,None,None
2,BeagleBone Black Wireless,US000011,Electronics,"October 06, 2016",https://certification.oshwa.org/us000011.html,BeagleBone Black Wireless,BeagleBoard.org Foundation,"October 06, 2016",United States of America,"Open hardware computer for makers, educators a...",...,[],Other,Other,CC BY,"[Electronics, Education]",[Computer],https://beagleboard.org/black-wireless,https://github.com/beagleboard/BeagleBone-blac...,None,None
3,ColorHug,UK000001,Electronics,"August 20, 2018",https://certification.oshwa.org/uk000001.html,ColorHug,Hughski Limited,"August 20, 2018",United Kingdom,A display colorimeter.,...,[],CERN,GPL,CC BY-SA,"[Electronics, Arts, Tool]","[color, calibration, display]",http://www.hughski.com/,https://hughski.com/faq.html#openhardware,None,None
4,HackEEG,US000027,Electronics,"October 08, 2016",https://certification.oshwa.org/us000027.html,HackEEG,"StarCat, LLC","October 08, 2016",United States of America,TI ADS1299 EEG SoC shield for Arduino Due.,...,[],Other,Other,CC BY-SA,"[Electronics, Robotics]","[EEG, Arduino, shield]",https://starcat.io/products/hackeeg-shield/,https://github.com/adamfeuer/hackeeg-shield,None,None


## 12. Texto técnico para cada proyecto

Empezamos con las transformaciones posteriores al scraping y esto ya es a parte de la captura. 
Vamos a crear un campo auxiliar llamado `technical_text` que no sustituirá ninguno de los campos anteriores sinó que será el campo que utilizaremos para hacer luego la busqueda sobre el nombre de la tecnología. Para rellenar este campo utilizaremos el nombre del proyecto + la descripción + los keywords.

In [17]:
# Unimos nombre, descripción y keywords en un único texto.
def construir_texto_tecnico(fila):
    partes = []

    for columna in ["project_name", "description"]:
        valor = fila.get(columna)
        if pd.notna(valor) and str(valor).strip():
            partes.append(str(valor))
    keywords = fila.get("keywords")
    if isinstance(keywords, list):
        partes.extend(str(k) for k in keywords)
    elif pd.notna(keywords) and str(keywords).strip():
        partes.append(str(keywords))
    return " ".join(partes)

if not df_oshwa.empty:
    df_oshwa["technical_text"] = df_oshwa.apply(construir_texto_tecnico, axis=1)
    display(df_oshwa[["uid", "project_name", "technical_text"]].head())

,uid,project_name,technical_text
0,US000001,OpenLog,OpenLog The SparkFun OpenLog is an open source...
1,US000051,blink(1) USB notification light,blink(1) USB notification light blink(1) is a ...
2,US000011,BeagleBone Black Wireless,BeagleBone Black Wireless Open hardware comput...
3,UK000001,ColorHug,ColorHug A display colorimeter. color calibrat...
4,US000027,HackEEG,HackEEG TI ADS1299 EEG SoC shield for Arduino ...


## 13. Tecnologías de interés

Aunqué hemos filtrado por `electrónica` no nos interesan todos los proyectos y se decidio filtrar aun más según el BOM publicado únicamente en los proyectos que tienen una tecnología que se pueda programar y tengan comunicación. No queremos proyectos demasiado faciles que solamente tengan LEDs, motores, sensores, etc.  Entonces, creamos un diccionario con estas tecnologias, como por ejemplo: ESP32, RS-485, Bluetooth, LoRa, etc. y para hacerlo lo más rápido es utilizar expresiones reculares con re.search() y re.INGNORECASE.

In [18]:
# Bibliografía:
#              - https://stackoverflow.com/questions/51487012/how-to-print-matched-keywords-from-two-lists
#              - https://stackoverflow.com/questions/66523520/python-how-to-use-list-of-keywords-to-search-for-a-string-in-a-text
#              - https://stackoverflow.com/questions/15650244/using-regex-in-python-2-7-3-to-search-text-and-output-matches
#              - https://stackoverflow.com/questions/15863066/how-to-match-a-whole-word-with-a-regular-expression
#              - https://docs.python.org/3/library/re.html

# Se adapta el patrón "lista de palabras -> expresión regular -> búsqueda" al vocabulario técnico de electrónica.

PATRONES_TECNOLOGIA = {
    "ESP32": [r"\bESP32(?:[- ][A-Z0-9]+)?\b"],
    "ESP8266": [r"\bESP8266\b"],
    "STM32": [r"\bSTM32[A-Z0-9-]*\b"],
    "RP2040": [r"\bRP2040\b", r"\bRaspberry\s+Pi\s+Pico\b"],
    "Raspberry Pi": [r"\bRaspberry\s+Pi\b"],
    "Arduino": [r"\bArduino\b"],
    "ATmega": [r"\bATmega[A-Z0-9-]*\b"],
    "ATtiny": [r"\bATtiny[A-Z0-9-]*\b"],
    "nRF52": [r"\bnRF52[A-Z0-9-]*\b", r"\bnRF528[0-9A-Z-]*\b"],
    "RISC-V": [r"\bRISC[- ]?V\b"],
    "FPGA": [r"\bFPGA\b"],
    "LoRa": [r"\bLoRa(?:WAN)?\b"],
    "CAN": [r"\bCAN\s*bus\b", r"\bCANbus\b"],
    "RS-485": [r"\bRS[- ]?485\b"],
    "RS-232": [r"\bRS[- ]?232\b"],
    "Ethernet": [r"\bEthernet\b"],
    "Wi-Fi": [r"\bWi[- ]?Fi\b"],
    "Bluetooth/BLE": [r"\bBluetooth\b", r"\bBLE\b", r"\bBluetooth\s+Low\s+Energy\b"],
    "Zigbee": [r"\bZigbee\b"],
    "USB": [r"\bUSB(?:[- ][A-Z0-9.]+)?\b"],
    "I2C": [r"\bI2C\b", r"\bI²C\b"],
    "SPI": [r"\bSPI\b"],
    "UART": [r"\bUART\b"],
    "GNSS/GPS": [r"\bGNSS\b", r"\bGPS\b"],
    "IMU": [r"\bIMU\b"]
}


def detectar_tecnologias(texto):
    """Devuelve las tecnologías del diccionario encontradas en el texto."""
    if texto is None or pd.isna(texto): return []
    texto = str(texto)
    encontradas = []
    for tecnologia, patrones in PATRONES_TECNOLOGIA.items():
        if any(re.search(patron, texto, flags=re.IGNORECASE) for patron in patrones):
            encontradas.append(tecnologia)
    return encontradas

if not df_oshwa.empty:
    df_oshwa["technology_mentions"] = df_oshwa["technical_text"].apply(detectar_tecnologias)
    df_oshwa["technology_count"] = df_oshwa["technology_mentions"].apply(len)
    display(df_oshwa[["uid", "project_name", "technology_mentions", "technology_count"]].head(15))

,uid,project_name,technology_mentions,technology_count
0,US000001,OpenLog,[Arduino],1
1,US000051,blink(1) USB notification light,[USB],1
2,US000011,BeagleBone Black Wireless,[],0
3,UK000001,ColorHug,[],0
4,US000027,HackEEG,[Arduino],1
5,FR000001,Smoothieware,[],0
6,ES000002,SAV MAKER I,[Arduino],1
7,US000028,OpenEVSE,[],0
8,NZ000001,OneRNG,[],0
9,MX000001,Byteman,[],0


Al mismo tiempo se podría conocer exactamente que chipsets se instalan con cada una de estas tecnologías y tener asi las referencias en el dataset en caso que se quiera comprar ese mismo componente. Si inspecionamos el BOM de los diferentes proyectos podemos ver que estas referencias utilizan una combinación de letras y números, por ejemplo: BME280, STM32G030, RP2040, DHT11, MCP2515, etc. y para detectarlas también se pueden utilizar expresiones regulares. El problema que tenemos es que no todo lo que encontremos con letras y números serán componentes reales asi que lo guardaremos en una columna llamada `candidate_components` y luego se podra hacer una limpieza para eliminar por ejemplo versiones, medidas o otros falsos positivos.

In [19]:
# No se consideran todavía componentes confirmados. Es una extracción exploratoria que se podrá limpiar en una fase posterior.
# Bibliografía:
#              - https://stackoverflow.com/questions/51487012/how-to-print-matched-keywords-from-two-lists
#              - https://www.geeksforgeeks.org/python/python-program-to-find-start-and-end-indices-of-all-words-in-a-string/
#              - https://docs.python.org/3/library/re.html#re.findall

PATRON_TOKEN_TECNICO = re.compile(
    r"\b"  # Inicio en un límite de palabra
    r"(?=[A-Za-z0-9._+-]{4,}\b)" # Al menos 4 caracteres alfanuméricos o símbolos . _ + - antes de un límite de palabra
    r"(?=[A-Za-z0-9._+-]*[A-Za-z])" # Al menos una letra
    r"(?=[A-Za-z0-9._+-]*\d)" # Al menos un dígito
    r"[A-Za-z0-9][A-Za-z0-9._+-]*" # Primer carácter alfanumérico, seguido de alfanuméricos o símbolos . _ + -
    r"\b") # Termina en un límite de palabra

def extraer_tokens_tecnicos(texto):
    if texto is None or pd.isna(texto): return []
    tokens = PATRON_TOKEN_TECNICO.findall(str(texto))

    # Quitamos duplicados conservando el orden de aparición.
    return list(dict.fromkeys(tokens))

if not df_oshwa.empty:
    df_oshwa["candidate_components"] = df_oshwa["technical_text"].apply(extraer_tokens_tecnicos)
    df_oshwa["candidate_token_count"] = df_oshwa["candidate_components"].apply(len)
    display(df_oshwa[["uid","project_name", "candidate_components"]].head(15))

,uid,project_name,candidate_components
0,US000001,OpenLog,[32GB]
1,US000051,blink(1) USB notification light,[]
2,US000011,BeagleBone Black Wireless,[]
3,UK000001,ColorHug,[]
4,US000027,HackEEG,[ADS1299]
5,FR000001,Smoothieware,[]
6,ES000002,SAV MAKER I,[]
7,US000028,OpenEVSE,[]
8,NZ000001,OneRNG,[]
9,MX000001,Byteman,[]


En los resultados anteriores podemos ver rápidamente como por ejemplo encontramos `32GB` como compoenente candidato pero sabemos que no lo es (es una medida de memoria). En el proyecto 4 en cambio encontramos `ADS1299` que es un componente real. Si hiciera falta esto ya se limpiará en una fase posterior. Por ahora nos interesa extraer todos los posibles candidatos unicamente.

La pregunta ahora es ¿y si hay varios proyectos que utilicen la misma tecnología, cómo lo podemos saber? Una solución sería contarlos pero perderiamos la referencia de en que proyecto se utilzia. Otra solución seria crear un diccionario con la tecnología como clave y la lista de proyectos como valor. Pero la solución que nos pareció más simple es directamente crear unos indicadores booleaneos para cada tipo de tecnología y proyecto, con lo que hacemos el siguiente apartado.

## 14. Indicadores booleanos para cada tecnología

Lo que queremos hacer es algo asi:

| project | ESP32 | STM32 | LoRa | CAN |
|---|---:|---:|---:|---:|
| Proyecto A | True | False | True | False |
| Proyecto B | False | True | False | True |
| Proyecto C | False | False | True | True |

Y con esto luego podremos hacer un análisis correlacionar para:
- calcular frecuencias de aparición de cada tecnología
- estudiar combinanciones de tecnologías que aparecen juntas
- construir vectores de características
- aplicar clusterización y clasificación
- desarrollos de sistemas de recomendacion basados en similitud

Aqui solo creamos las variables y el análisis o entrenamiento de modelos queda fuera del alcance actual de esta práctica.

In [20]:
# Creamos una variable booleana para cada tecnología detectada, equivalente a comprobar si un elemento pertenece a la lista de coincidencias obtenida para cada registro.
def usa_tecnologia(lista, tecnologia):
    return tecnologia in lista

if not df_oshwa.empty:
    for tecnologia in PATRONES_TECNOLOGIA:
        nombre_columna = ("usa_" + re.sub(r"[^a-z0-9]+","_", tecnologia.casefold()).strip("_"))
        df_oshwa[nombre_columna] = df_oshwa["technology_mentions"].apply(usa_tecnologia,args=(tecnologia,))
    columnas_uses = [columna for columna in df_oshwa.columns if columna.startswith("usa_")]
    display(df_oshwa[["uid", "project_name"] + columnas_uses[:10]].head())

,uid,project_name,usa_esp32,usa_esp8266,usa_stm32,usa_rp2040,usa_raspberry_pi,usa_arduino,usa_atmega,usa_attiny,usa_nrf52,usa_risc_v
0,US000001,OpenLog,False,False,False,False,False,True,False,False,False,False
1,US000051,blink(1) USB notification light,False,False,False,False,False,False,False,False,False,False
2,US000011,BeagleBone Black Wireless,False,False,False,False,False,False,False,False,False,False
3,UK000001,ColorHug,False,False,False,False,False,False,False,False,False,False
4,US000027,HackEEG,False,False,False,False,False,True,False,False,False,False


In [21]:
# Comprobamos las tecnologías que aparecen en la muestra de proyectos que hemos extraído anterirormente.
if not df_oshwa.empty:
    frecuencia_tecnologias = (df_oshwa["technology_mentions"].explode().dropna().value_counts().rename_axis("technology").reset_index(name="projects"))
    display(frecuencia_tecnologias)

,technology,projects
0,I2C,200
1,Arduino,162
2,USB,123
3,SPI,62
4,Raspberry Pi,57
5,Wi-Fi,50
6,Bluetooth/BLE,40
7,GNSS/GPS,38
8,ATmega,24
9,UART,22


In [22]:
# Comprobamos como fue la extracción comprobando nº de registros, nº de UID, columnas, valores nulos, UID duplicados, etc. 

print("Registros:", len(df_oshwa))

if not df_oshwa.empty:
    print("UID diferentes:", df_oshwa["uid"].nunique())
    print("UID duplicados:", df_oshwa["uid"].duplicated().sum())
    print("\nColumnas obtenidas:")
    print(df_oshwa.columns.tolist())
    print("\nValores nulos por columna:")
    display(df_oshwa.isna().sum().to_frame("nulos"))
else:
    print("El DataFrame está vacío.")

Registros: 1000
UID diferentes: 1000
UID duplicados: 0

Columnas obtenidas:
['project_name_list', 'uid', 'primary_project_type', 'certification_date_list', 'detail_url', 'project_name', 'author', 'certification_date', 'country', 'description', 'version', 'previous_versions', 'hardware_license', 'software_license', 'documentation_license', 'project_types', 'keywords', 'project_website', 'documentation_url', 'certification_svg_url', 'certification_png_url', 'technical_text', 'technology_mentions', 'technology_count', 'candidate_components', 'candidate_token_count', 'usa_esp32', 'usa_esp8266', 'usa_stm32', 'usa_rp2040', 'usa_raspberry_pi', 'usa_arduino', 'usa_atmega', 'usa_attiny', 'usa_nrf52', 'usa_risc_v', 'usa_fpga', 'usa_lora', 'usa_can', 'usa_rs_485', 'usa_rs_232', 'usa_ethernet', 'usa_wi_fi', 'usa_bluetooth_ble', 'usa_zigbee', 'usa_usb', 'usa_i2c', 'usa_spi', 'usa_uart', 'usa_gnss_gps', 'usa_imu']

Valores nulos por columna:


,nulos
project_name_list,0
uid,0
primary_project_type,0
certification_date_list,0
detail_url,0
project_name,0
author,0
certification_date,0
country,0
description,0


## 15. Preparar la lista para guardar el CSV

CSV es un formato tabular y antes de exportar hay que convertir las listas en cadenas separadas por |, por ejemplo `Electronics|IOT|Robotics`. No limpiamos el contenido, únicamente cambiamos su representación para poder almacenarlo en una celda de un CSV.

In [23]:
# Bibliografía:
#          - https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html
#          - https://pandas.pydata.org/docs/reference/api/pandas.Series.apply.html
#          - https://docs.python.org/3/reference/expressions.html#lambda
#          - https://docs.python.org/3/library/functions.html#isinstance
#          - https://docs.python.org/3/library/stdtypes.html#str.join
#          - https://ipython.readthedocs.io/en/latest/api/generated/IPython.display.html

def convertir_lista_a_texto(valor):
    if isinstance(valor, list):
        return "|".join(valor)
    return valor

df_export = df_oshwa.copy()
columnas_lista = ["previous_versions", "project_types", "keywords", "technology_mentions", "candidate_components"]

for columna in columnas_lista:
    if columna in df_export.columns:
        df_export[columna] = df_export[columna].apply(convertir_lista_a_texto)
display(df_export.head())
# La versión pública excluye correos también si aparecen en texto libre.
PATRON_CORREO = re.compile(r"[A-Za-z0-9.!#$%&'*+/=?^_`{|}~-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)+")
def excluir_correos(valor):
    return PATRON_CORREO.sub("[correo omitido]", valor) if isinstance(valor, str) else valor
for columna in df_export.columns:
    df_export[columna] = df_export[columna].apply(excluir_correos)


,project_name_list,uid,primary_project_type,certification_date_list,detail_url,project_name,author,certification_date,country,description,...,usa_ethernet,usa_wi_fi,usa_bluetooth_ble,usa_zigbee,usa_usb,usa_i2c,usa_spi,usa_uart,usa_gnss_gps,usa_imu
0,OpenLog,US000001,Electronics,"October 01, 2016",https://certification.oshwa.org/us000001.html,OpenLog,Nathan Seidle,"October 01, 2016",United States of America,The SparkFun OpenLog is an open source data lo...,...,False,False,False,False,False,False,False,False,False,False
1,blink(1) USB notification light,US000051,Electronics,"October 04, 2016",https://certification.oshwa.org/us000051.html,blink(1) USB notification light,ThingM,"October 04, 2016",United States of America,blink(1) is a programmable USB notification li...,...,False,False,False,False,True,False,False,False,False,False
2,BeagleBone Black Wireless,US000011,Electronics,"October 06, 2016",https://certification.oshwa.org/us000011.html,BeagleBone Black Wireless,BeagleBoard.org Foundation,"October 06, 2016",United States of America,"Open hardware computer for makers, educators a...",...,False,False,False,False,False,False,False,False,False,False
3,ColorHug,UK000001,Electronics,"August 20, 2018",https://certification.oshwa.org/uk000001.html,ColorHug,Hughski Limited,"August 20, 2018",United Kingdom,A display colorimeter.,...,False,False,False,False,False,False,False,False,False,False
4,HackEEG,US000027,Electronics,"October 08, 2016",https://certification.oshwa.org/us000027.html,HackEEG,"StarCat, LLC","October 08, 2016",United States of America,TI ADS1299 EEG SoC shield for Arduino Due.,...,False,False,False,False,False,False,False,False,False,False


## 16. Guardamos el CSV final

In [24]:
# Guardamos el dataset con DataFrame.to_csv()
# Bibliografía:
#              - https://www.geeksforgeeks.org/pandas/saving-a-pandas-dataframe-as-a-csv/
#              - https://www.geeksforgeeks.org/python/working-csv-files-python/

from pathlib import Path

# Funciona al abrir Jupyter desde la raíz del repositorio o desde source/.
raiz_repositorio = Path.cwd().resolve()

if raiz_repositorio.name == "source":
    raiz_repositorio = raiz_repositorio.parent
if not (raiz_repositorio / "source").is_dir():
    raise RuntimeError("Abre Jupyter desde Practica1 o desde su carpeta source.")

carpeta_dataset = raiz_repositorio / "dataset"
carpeta_dataset.mkdir(exist_ok=True)
nombre_csv = carpeta_dataset / "oshwa_electronics_technologies_dataset_v1.csv"

# No sustituir el dataset anterior si faltan fichas o hay duplicados.
if df_export.empty or len(df_export) != len(proyectos_seleccionados):
    raise RuntimeError("Extracción incompleta: no se sustituye el CSV anterior")
if df_export["uid"].duplicated().any():
    raise RuntimeError("UID duplicados: no se sustituye el CSV anterior")

archivo_temporal = nombre_csv.with_suffix(".csv.tmp")
df_export.to_csv(archivo_temporal, index=False, encoding="utf-8")
archivo_temporal.replace(nombre_csv)

print("Dataset guardado:", nombre_csv)
print("Número de registros:", len(df_export))

Dataset guardado: C:\Users\Mihail\OneDrive\0000 Master Data Science\0003 Primero 26-27\0002 Tipologia de datos\Practica1\dataset\oshwa_electronics_technologies_dataset_v1.csv
Número de registros: 1000


## Resumen de la primera versión del scraper

En esta primera versión se hizo el proceso completo de navegación:

- Se intenta comprobar `robots.txt`.
- Se utiliza un `User-Agent` identificable.
- Se utiliza `requests.Session()`.
- Se añaden pausas entre peticiones.
- Se detiene el proceso si aparecen respuestas `403` o `429`.
- Se descarga el directorio real de proyectos certificados de OSHWA.
- Se localiza la tabla HTML.
- Se recorren automáticamente sus filas.
- Se descubren los enlaces a las fichas individuales.
- Se filtran inicialmente los proyectos cuyo tipo principal es `Electronics`.
- Se entra automáticamente en cada ficha que se ha descubierto.
- Se extraen los campos textuales, categóricos, fechas, listas y enlaces.
- Se genera un DataFrame.
- Se exporta el resultado a CSV.

Para las pruebas de desarrollo limitaos el número de proyectos a 20 pero se puede aumentar al numero que se quiera ya que el script se deja preparado para ello.

Es importante diferenciar entre **datos extraídos de OSHWA** y **variables derivadas posteriormente** ya que hay varias columnas, como por ejemplo `technology_mentions`, `candidate_components` y `usa_*` que las hicimos nosotros porque las hemos considerado interesantes para futuros análisis del dataset no porque sean parte de la información original.